In [ ]:
# 🩻 ChestNet — ResNet50 Transfer Learning (Full A→Z Pipeline)

**Dataset:** NIH ChestX-ray14 random sample (`sample_labels.csv`, 5,606 images) · **multi-label**
**Model:** ResNet50, ImageNet-pretrained, fully fine-tuned · **Framework:** PyTorch
**Settings:** follow `configs/base.yaml` + `configs/resnet50.yaml` so the comparison with the other models is fair.

| Step | What it does |
|---|---|
| 0 | Install / imports |
| 1 | Config ⚙️ |
| 2 | Helper functions 🧰 |
| 3 | Load + clean labels, quick EDA 🏷️ |
| 4 | Patient-level train/val/test split ✂️ |
| 5 | Resize + cache images (run once) 🖼️ |
| 6 | Dataset, augmentations, DataLoaders 🗂️ |
| 7 | Sanity check ✅ |
| 8 | ResNet50 model 🧠 |
| 9 | Training 🏋️ |
| 10 | Evaluation on the test set 📈 |
| 11 | Single-image prediction + Grad-CAM 🔥 |

> 💡 Run the cells **top to bottom**. Everything is written to `outputs/resnet50/` (outside the `ChestNet` git folder).

In [ ]:
## 0️⃣ Install & imports

In [ ]:
import json
import os
import random
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision import transforms as T

from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                             precision_score, recall_score, f1_score,
                             multilabel_confusion_matrix)

print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
## 1️⃣ Config ⚙️
Change paths and hyperparameters **here only**.

In [ ]:
# ---------- Paths ----------
from pathlib import Path

# ---------- Paths ----------
ROOT = Path("/content/drive/MyDrive/ChestNet")
DATA_DIR = ROOT / "data"
RAW_IMAGE_DIR = DATA_DIR / "raw_images"          # put images_001 ... images_012 folders here
IMAGE_DIR = DATA_DIR / "images_256"              # resized copies are written here (step 3)
CSV_PATH = DATA_DIR / "Data_Entry_2017.csv"      # NIH metadata file
SPLIT_DIR = DATA_DIR / "splits"                  # train/val/test CSVs (step 4)
STATS_PATH = DATA_DIR / "norm_stats.json"        # mean/std (step 5)
OUT_DIR = ROOT / "outputs" / "ResNet"
DATASET_DIR = IMAGE_DIR            # 📁 images_256 — where your PNGs actually are
MAX_IMAGES_PER_CLASS = 3000        # 🧮 cap for "No Finding" (pick what you like)        # checkpoints, plots, reports
# ---------- Labels (14 NIH diseases) ----------
CLASSES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass",
    "Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia",
]
NUM_CLASSES = len(CLASSES)

# ---------- Image ----------
RESIZE_TO = 256      # size stored on disk (step 3)
IMG_SIZE = 224       # size fed to the network
IN_CHANNELS = 1      # grayscale X-rays

# ---------- Split ----------
TEST_FRAC = 0.20
VAL_FRAC = 0.10
SEED = 42

# ---------- Training ----------
BATCH_SIZE = 32
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7               # early-stopping patience (epochs)
MAX_POS_WEIGHT = 50.0      # cap for class-imbalance weights

# ⚠️ In Jupyter on Windows, num_workers > 0 usually crashes (classes defined in a
# notebook can't be sent to worker processes). Keep 0 in the notebook.
NUM_WORKERS = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"  # mixed precision only on GPU

OUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE)

In [ ]:
## 2️⃣ Helper functions 🧰

In [ ]:
def set_seed(seed: int = SEED):
    # Same seed -> same split, same augmentations, same initial head weights
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True   # fastest conv algorithm for a fixed image size


def make_labels(df: pd.DataFrame) -> pd.DataFrame:
    # "Effusion|Mass" -> one 0/1 column per class (multi-hot)
    df = df.copy()
    labels = df["Finding Labels"].str.split("|")
    for c in CLASSES:
        df[c] = labels.apply(lambda l: int(c in l))
    # "No Finding" must not co-exist with a disease
    disease_cols = [c for c in CLASSES if c != "No Finding"]
    if "No Finding" in CLASSES:
        df.loc[df[disease_cols].sum(axis=1) > 0, "No Finding"] = 0
    return df


def compute_aucs(labels: np.ndarray, probs: np.ndarray) -> np.ndarray:
    # ROC-AUC per class; NaN if a class has only positives or only negatives in this set
    aucs = []
    for i in range(labels.shape[1]):
        if len(np.unique(labels[:, i])) < 2:
            aucs.append(np.nan)
        else:
            aucs.append(roc_auc_score(labels[:, i], probs[:, i]))
    return np.array(aucs)


def plot_history(history: pd.DataFrame, save_path=None):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(history["epoch"], history["train_loss"], label="train")
    ax[0].plot(history["epoch"], history["val_loss"], label="val")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(history["epoch"], history["train_auc"], label="train")
    ax[1].plot(history["epoch"], history["val_auc"], label="val")
    ax[1].set_title("Macro AUC"); ax[1].set_xlabel("epoch"); ax[1].legend()
    ax[2].plot(history["epoch"], history["lr"])
    ax[2].set_title("Learning rate"); ax[2].set_xlabel("epoch")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
    plt.show()


set_seed()